In [1]:
pip install opencv-python pytesseract pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import cv2
import pytesseract
import pandas as pd

# Si estás en Windows, descomenta y configura esta línea:
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

def extraer_tabla_desde_imagen(ruta_imagen):
    # Leer imagen
    imagen = cv2.imread(ruta_imagen)
    
    # Convertir a escala de grises
    gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)

    # Aplicar umbral para mejorar el contraste
    _, umbral = cv2.threshold(gris, 150, 255, cv2.THRESH_BINARY_INV)

    # Encontrar contornos (puede ayudar a ubicar celdas)
    contornos, _ = cv2.findContours(umbral, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Opcional: dibujar contornos (para depurar)
    # cv2.drawContours(imagen, contornos, -1, (0,255,0), 1)
    # cv2.imshow("Contornos", imagen)
    # cv2.waitKey(0)

    # Aplicar OCR a la imagen completa (o usar ROI por celda)
    texto = pytesseract.image_to_string(imagen, config='--psm 6')

    # Dividir líneas y columnas (esto depende del formato visual de la tabla)
    filas = [linea.strip() for linea in texto.split('\n') if linea.strip()]
    datos = [fila.split() for fila in filas]  # o usar split('\t') si el OCR reconoce tabs

    # Convertir a DataFrame si todas las filas tienen el mismo número de columnas
    max_cols = max(len(f) for f in datos)
    datos_uniformes = [fila + [''] * (max_cols - len(fila)) for fila in datos]  # rellenar vacíos
    df = pd.DataFrame(datos_uniformes)

    return df

# Ejemplo de uso
ruta = 'images/imagen1.png'  # Cambia esto por tu imagen
df_tabla = extraer_tabla_desde_imagen(ruta)
print(df_tabla)


        0      1       2       3          4       5       6          7  \
0   Fecha  Abrir    Max.    Min.     Cerrar       O  Cierre   ajustado   
1       8    abr    2025  153,57     154,44  145,21  146,58     146,58   
2       T    abr    2025  143,39     154,93  142,66  149,24     149,24   
3       A    abr    2025  149,90     153,09  147,54  147,74     147,74   
4       3    abr    2025  152,84     154,69  152,18  152,63     152,63   
5    2abr   2025  156,96  160,27     156,53  158,86  158,86  17113.300   
6       1    abr    2025  155,30     160,08  155,26  158,88     158,88   
7      31    mar    2025  154,81     157,13  152,21  156,23     156,23   
8      28    mar    2025  162,36     163,81  155,34  156,06     156,06   
9      27    mar    2025  166,71     167,44  163,85  164,08     164,08   
10     26    mar    2025  171,30     171,94  166,86  167,14     167,14   
11     25    mar    2025   17148     172,91  170,55  172,79     172,79   
12     24    mar    2025  169,26     1